# ORC Basics — 05: ORC vs Parquet

The definitive comparison. Both are columnar formats but with different
strengths. This notebook benchmarks them head-to-head on identical data.

Topics: storage size, write/read speed, column pruning, filter pushdown,
schema evolution, ecosystem support, when to use each.


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob as glib
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/orc_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('orc-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 13:26:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2


## Step 1 — Setup: Same Data, Both Formats



In [2]:

import random,datetime,pathlib
random.seed(42); N=400_000
rows=[]
base=datetime.date(2023,1,1)
CATS=["Electronics","Clothing","Books","Food","Sports","Furniture"]
CTRS=["US","UK","DE","FR","JP","CA","AU","BR"]
for i in range(N):
    qty=random.randint(1,10); price=round(random.uniform(5,1000),2)
    d=base+datetime.timedelta(days=random.randint(0,365))
    rows.append((i+1,random.randint(1,50000),random.choice(CATS),random.choice(CTRS),
                 qty,price,round(qty*price,2),random.choice(["pending","confirmed","shipped"]),d))
df_bench=spark.createDataFrame(rows,["order_id","customer_id","category","country","quantity","price","revenue","status","order_date"])

ORC_B=f"{DATA_DIR}/bench_orc"; PQ_B=f"{DATA_DIR}/bench_parquet"
df_bench.write.mode("overwrite").option("compression","snappy").orc(ORC_B)
df_bench.write.mode("overwrite").option("compression","snappy").parquet(PQ_B)

orc_mb=sum(pathlib.Path(f).stat().st_size for f in glib.glob(f"{ORC_B}/*.orc"))/1024/1024
pq_mb =sum(pathlib.Path(f).stat().st_size for f in glib.glob(f"{PQ_B}/*.parquet"))/1024/1024
print(f"Same data, snappy compression:")
print(f"  ORC     : {orc_mb:.1f} MB")
print(f"  Parquet : {pq_mb:.1f} MB  (ratio: {orc_mb/pq_mb:.2f}x)")


26/04/10 13:26:49 WARN TaskSetManager: Stage 0 contains a task of very large size (3790 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 13:26:52 WARN TaskSetManager: Stage 1 contains a task of very large size (3790 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Same data, snappy compression:
  ORC     : 6.6 MB
  Parquet : 8.0 MB  (ratio: 0.83x)


## Step 2 — Read Performance Benchmark



In [3]:

queries=[
    ("Full scan + agg",   lambda p,f: spark.read.format(f).load(p).agg(F.sum("revenue")).collect()),
    ("Filter + count",    lambda p,f: spark.read.format(f).load(p).filter(F.col("category")=="Electronics").count()),
    ("Select 2 cols",     lambda p,f: spark.read.format(f).load(p).select("revenue","category").agg(F.sum("revenue")).collect()),
    ("GroupBy 6 groups",  lambda p,f: spark.read.format(f).load(p).groupBy("category").agg(F.sum("revenue")).collect()),
    ("Multi-filter",      lambda p,f: spark.read.format(f).load(p).filter((F.col("category")=="Electronics")&(F.col("revenue")>500)).count()),
]
print(f"{'Query':<28} {'ORC':>8} {'Parquet':>10} {'Winner':>10}")
print("-"*60)
for label,fn in queries:
    to,tp=[],[]
    for _ in range(3):
        t0=time.time(); fn(ORC_B,"orc");    to.append(time.time()-t0)
        t0=time.time(); fn(PQ_B,"parquet"); tp.append(time.time()-t0)
    t_o,t_p=sorted(to)[1],sorted(tp)[1]
    winner="ORC" if t_o<t_p else "Parquet"
    speedup=max(t_o,t_p)/min(t_o,t_p)
    print(f"  {label:<26} {t_o:>6.3f}s  {t_p:>8.3f}s  {winner}({speedup:.1f}x)")


Query                             ORC    Parquet     Winner
------------------------------------------------------------


  Full scan + agg             0.414s     0.438s  ORC(1.1x)
  Filter + count              0.333s     0.353s  ORC(1.1x)
  Select 2 cols               0.291s     0.315s  ORC(1.1x)
  GroupBy 6 groups            0.334s     0.390s  ORC(1.2x)
  Multi-filter                0.273s     0.318s  ORC(1.2x)


## Step 3 — Schema Evolution Comparison



In [4]:

print("Schema Evolution:")
print()
print("  Parquet:")
print("    ADD column   : mergeSchema=True → old files return null  ✅")
print("    RENAME column: breaks reading old files                  ❌")
print("    Type widen   : supported (int→long)                      ✅")
print()
print("  ORC:")
print("    ADD column   : schema merging supported                  ✅")
print("    RENAME column: breaks reading old files                  ❌")
print("    Type widen   : limited support                           ⚠️")
print()
print("Neither ORC nor Parquet has first-class schema evolution.")
print("Use Iceberg or Delta for production tables with evolving schemas.")


Schema Evolution:

  Parquet:
    ADD column   : mergeSchema=True → old files return null  ✅
    RENAME column: breaks reading old files                  ❌
    Type widen   : supported (int→long)                      ✅

  ORC:
    ADD column   : schema merging supported                  ✅
    RENAME column: breaks reading old files                  ❌
    Type widen   : limited support                           ⚠️

Neither ORC nor Parquet has first-class schema evolution.
Use Iceberg or Delta for production tables with evolving schemas.


## Step 4 — When to Use Each



In [5]:

print("""
Decision guide:

Use ORC when:
  ✅ Hive ecosystem — ORC is Hive's native format
  ✅ Amazon Athena / EMR — strong ORC optimizations
  ✅ Need bloom filters for equality lookups
  ✅ You need SUM statistics (ORC stores sum; Parquet does not)
  ✅ Legacy Hive ACID tables
  ✅ String-heavy data — ORC dictionary sometimes better

Use Parquet when:
  ✅ Spark / Trino / Flink / BigQuery / Snowflake — widest support
  ✅ Iceberg or Delta table format (both default to Parquet)
  ✅ ZSTD compression (better ratio for modern workloads)
  ✅ Cloud storage (S3, GCS, ADLS) — Parquet more common
  ✅ New projects — Parquet is the modern standard

Both are:
  ✅ Columnar (column pruning + predicate pushdown)
  ✅ Compressed (snappy/zlib/zstd available)
  ✅ Self-describing (schema embedded)
  ✅ Splittable (parallel reads)

Performance: roughly equivalent — differences are workload-specific.
Test with YOUR data and queries before choosing.
""")



Decision guide:

Use ORC when:
  ✅ Hive ecosystem — ORC is Hive's native format
  ✅ Amazon Athena / EMR — strong ORC optimizations
  ✅ Need bloom filters for equality lookups
  ✅ You need SUM statistics (ORC stores sum; Parquet does not)
  ✅ Legacy Hive ACID tables
  ✅ String-heavy data — ORC dictionary sometimes better

Use Parquet when:
  ✅ Spark / Trino / Flink / BigQuery / Snowflake — widest support
  ✅ Iceberg or Delta table format (both default to Parquet)
  ✅ ZSTD compression (better ratio for modern workloads)
  ✅ Cloud storage (S3, GCS, ADLS) — Parquet more common
  ✅ New projects — Parquet is the modern standard

Both are:
  ✅ Columnar (column pruning + predicate pushdown)
  ✅ Compressed (snappy/zlib/zstd available)
  ✅ Self-describing (schema embedded)
  ✅ Splittable (parallel reads)

Performance: roughly equivalent — differences are workload-specific.
Test with YOUR data and queries before choosing.



## Summary



In [6]:

print("""
Head-to-head summary:
  Feature            ORC              Parquet
  ───────────────────────────────────────────
  Stripe/Row group   64 MB default    128 MB default
  Index granularity  10K rows         page (~1 MB)
  Statistics         min/max/sum/count  min/max/null
  Bloom filters      ✅ Built-in      ❌ Not standard
  Hive support       ✅ Native        ✅ Good
  Spark support      ✅               ✅
  Trino/Athena       ✅               ✅
  ZSTD compression   ✅               ✅
  Schema evolution   ⚠️               ⚠️
  Default in Delta   ❌               ✅
  Default in Iceberg ❌               ✅
""")



Head-to-head summary:
  Feature            ORC              Parquet
  ───────────────────────────────────────────
  Stripe/Row group   64 MB default    128 MB default
  Index granularity  10K rows         page (~1 MB)
  Statistics         min/max/sum/count  min/max/null
  Bloom filters      ✅ Built-in      ❌ Not standard
  Hive support       ✅ Native        ✅ Good
  Spark support      ✅               ✅
  Trino/Athena       ✅               ✅
  ZSTD compression   ✅               ✅
  Schema evolution   ⚠️               ⚠️
  Default in Delta   ❌               ✅
  Default in Iceberg ❌               ✅

